# Chapter 8: Transactions
## *Designing Data-Intensive Applications* — Türkçe Açıklamalar

> *"We believe it is better to have application programmers deal with performance problems due to overuse of transactions as bottlenecks arise, rather than always coding around the lack of transactions."*  
> — James Corbett et al., "Spanner: Google's Globally-Distributed Database" (2012)

---

Bu notebook, "Designing Data-Intensive Applications" kitabının 8. bölümünü (Transactions / İşlemler) detaylı biçimde Türkçe açıklamaktadır. Tüm terimler İngilizce orijinal halleriyle kullanılmakta; kavramlar, örnekler ve uygulama detayları eksiksiz ele alınmaktadır.

## 1. Neden Transaction'lara İhtiyaç Duyarız?

Gerçek dünya veri sistemlerinde pek çok şey yanlış gidebilir:

- **Donanım veya yazılım hatası**: Database yazılımı ya da donanımı bir write işlemi sırasında bile çöküşe geçebilir.
- **Uygulama crash'i**: Uygulama bir dizi operasyonun ortasında çökebilir.
- **Ağ kesintisi**: Network, uygulamayı database'den veya bir database node'unu diğerinden koparabilir.
- **Eş zamanlı yazma çakışmaları**: Birden fazla client aynı anda aynı veriyi yazarak birbirinin değişikliklerini silebilir.
- **Kısmi güncelleme okuma**: Bir client, henüz tamamlanmamış bir güncellemenin ortasındaki veriyi okuyabilir.
- **Race condition'lar**: Client'lar arasındaki yarış koşulları beklenmedik hatalara yol açabilir.

Bu tür hataları tolere edebilen güvenilir bir sistem inşa etmek son derece karmaşıktır. Pek çok olası başarısızlık senaryosunu dikkatle düşünmek ve uygulanan çözümlerin gerçekten işe yaradığını titizlikle test etmek gerekir.

### Transaction Nedir?

Onlarca yıldır **transaction** (işlem), bu sorunları basitleştirmenin en temel mekanizması olarak kullanılmaktadır. Bir transaction, bir uygulamanın birden fazla read (okuma) ve write (yazma) işlemini tek bir **logical unit** (mantıksal birim) olarak gruplandırmasının yoludur.

Kavramsal olarak, bir transaction içindeki tüm read'ler ve write'lar tek bir operasyon olarak yürütülür:
- Tüm transaction başarılı olursa → **commit** (kayıt)
- Herhangi bir adım başarısız olursa → **abort** veya **rollback** (geri alma)

Transaction başarısız olduğunda uygulama güvenle yeniden deneyebilir. Bu sayede **partial failure** (kısmi başarısızlık) konusunda endişelenmek gerekmez: ya her şey gerçekleşir, ya da hiçbir şey.

### Transaction'ların Amacı

Transaction'lar doğanın bir yasası değildir; belirli bir amaç için yaratılmışlardır: database'e erişen uygulamalar için **programming model'i (programlama modelini) basitleştirmek**.

Transaction'ları kullanmak, uygulamanın belirli olası hata senaryolarını ve concurrency sorunlarını görmezden gelmesine izin verir; çünkü bu sorunlarla başa çıkmak database'in sorumluluğundadır. Bu güvencelere **safety guarantees** (güvenlik güvenceleri) denir.

Her uygulama transaction gerektirmez; zaman zaman daha zayıf transactional garantiler kullanmak veya bunlardan tamamen vazgeçmek avantajlı olabilir (örneğin daha iyi performance veya daha yüksek availability için). Ancak transaction'lar ciddi sorunları önleyebilir — örneğin İngiltere'deki **Post Office Horizon Scandal**'ın teknik nedeni, altta yatan muhasebe sisteminde ACID transaction'larının olmamasıydı.

## 2. Transaction Tam Olarak Nedir? (Tarihsel Arka Plan)

Günümüzdeki ilişkisel database'lerin neredeyse tamamı ve bazı nonrelational database'ler transaction'ları destekler. Çoğu, 1975 yılında IBM tarafından geliştirilen ilk SQL database'i olan **IBM System R** tarafından tanıtılan stili takip eder.

Bu temel fikir 50 yıldır neredeyse hiç değişmedi. MySQL, PostgreSQL, Oracle, SQL Server gibi sistemlerdeki transaction desteği, System R'inkiyle şaşırtıcı derecede benzerdir.

### NoSQL Dönemi ve Transaction'lar

2000'lerin sonunda **NoSQL** database'leri popülerlik kazandı. Bu sistemler, yeni veri modelleri sunmak ve replication/sharding'i varsayılan olarak dahil etmek amacıyla relational database'lerin üstünde olmayı hedefledi. Bu süreçte transaction'lar büyük bedel ödedi: bu nesil database'lerin birçoğu transaction'ları tamamen terk etti ya da kelimeyi önceden anlaşılan anlamından çok daha zayıf bir garanti kümesini tanımlamak için yeniden tanımladı.

### NewSQL Dönemi ve Geri Dönüş

NoSQL distributed database'leri etrafındaki hype, transaction'ların doğası gereği ölçeklenmez olduğuna ve büyük ölçekli sistemlerin performans ve availability için bunları terk etmesi gerektiğine dair yaygın bir inanca yol açtı. Ancak bu inanç yanlış çıktı.

**NewSQL** olarak adlandırılan **CockroachDB**, **TiDB**, **Spanner**, **FoundationDB** ve **YugabyteDB** gibi database'ler, büyük veri hacimlerine ve yüksek throughput'a ölçeklenebilen transactional sistemler olabildiğini kanıtladı. Bu sistemler, sharding'i güçlü ACID garantileri sağlamak üzere **consensus protocol**'leriyle birleştirir.

## 3. ACID Özellikleri

Transaction'ların sağladığı güvenlik garantileri, çoğunlukla iyi bilinen **ACID** kısaltmasıyla tanımlanır:

| Harf | İngilizce | Açıklama |
|------|-----------|----------|
| **A** | Atomicity | Ya hep ya hiç |
| **C** | Consistency | Veri her zaman geçerli bir durumda |
| **I** | Isolation | Eş zamanlı transaction'lar birbirini görmez |
| **D** | Durability | Commit edilen veri kaybolmaz |

Bu terim, 1983 yılında Theo Härder ve Andreas Reuter tarafından database'lerdeki fault-tolerance (hata toleransı) mekanizmalarına yönelik kesin bir terminoloji oluşturma çabası içinde ortaya konmuştur.

> **Önemli Not**: Pratikte, bir database'in ACID uygulaması diğerinin uygulamasına eşit değildir. ACID, maalesef büyük ölçüde bir pazarlama terimi haline gelmiştir.

ACID kriterlerini karşılamayan sistemler bazen **BASE** olarak adlandırılır:
- **B**asically **A**vailable (Temel Düzeyde Erişilebilir)
- **S**oft state (Yumuşak Durum)
- **E**ventual consistency (Nihai Tutarlılık)

BASE, ACID'den de belirsizdir. BASE'in tek anlamlı tanımı "ACID değil" gibi görünmektedir (yani neredeyse istediğiniz her şeyi ifade edebilir).

---

### 3.1 Atomicity (Atomiklik)

Genel olarak **atomic**, daha küçük parçalara bölünemeyen bir şeyi ifade eder. Bilişimin farklı alanlarında benzer ama ince farklılıklar taşıyan anlamlara gelir:

- **Multithreaded programming'de**: Bir thread atomik bir operasyon yürütüyorsa, başka bir thread, o operasyonun yarı tamamlanmış sonucunu göremez. Sistem ya operasyondan önceki durumda ya da sonrasındaki durumda olabilir — hiçbir zaman ikisi arasında bir yerde olamaz.

- **ACID bağlamında**: Atomicity, concurrency ile ilgili **değildir**. Bunun yerine, bir thread yarı tamamlanmış bir write işleminde çöküşe geçerse ne olacağını tanımlar.

ACID bağlamında atomicity şu şekilde çalışır: Bir transaction birden fazla write içeriyorsa ve bu write'lardan biri başarısız olursa (ağ kesintisi, disk dolması, constraint ihlali vb.), transaction **abort** edilir ve o ana kadar yapılan write'lar geri alınır. Uygulama, transaction'ın gerçekleşmediğinden emin olarak güvenle işlemi yeniden deneyebilir.

Bu özelliğe **abortability** (iptal edilebilirlik) de denebilir. Yarı tamamlanmış bir transaction'ın hangi değişikliklerinin geçerli kılınacağı ve hangilerinin silineceği konusunda endişelenmeden güvenle yeniden deneme yapılabilmesini sağlar.

---

### 3.2 Consistency (Tutarlılık)

**Consistency** kelimesi veri sistemlerinde aşırı yüklüdür ve farklı bağlamlarda farklı anlamlara gelir. ACID bağlamındaki consistency, veritabanının her zaman "iyi bir durumda" olması gerektiğini, yani veriye ilişkin belirli ifadelerin (invariants/değişmezlerin) her zaman doğru olması gerektiğini ifade eder.

Örneğin bir muhasebe sisteminde, credits ve debits her zaman dengelenmelidir. Bir transaction bu invariant'ı başlangıçta ve sonunda sağlayarak çalışırsa, veritabanının her zaman tutarlı olacağı garanti edilir.

**Önemli bir nokta**: Consistency, uygulamanın doğruluğuna bağlıdır. Database, invariant'larınızı ihlal eden veriler yazmanızı engelleyemez (yabancı anahtar kısıtlamaları veya uniqueness kısıtlamaları gibi belirli türdeki invariant'lar için database yardımcı olabilse de). Bu, ACID'deki C'nin esas olarak diğer üç özellik (A, I, D) gibi database'in bir özelliği olmadığı, uygulamanın bir özelliği olduğu anlamına gelir.

---

### 3.3 Isolation (İzolasyon)

**Isolation**, aynı anda birden fazla kullanıcı eş zamanlı olarak aynı database'i okuyup yazıyorsa, eş zamanlı çalışan transaction'ların birbirini etkilememesi gerektiği anlamına gelir.

Klasik bağlamıyla; iki transaction eş zamanlı çalışıyorsa, commit ettiğinizde sonuç, birbirini takip eden (sequential) yürütme gibi görünmelidir — sanki sadece birisi vardı ve tamamladıktan sonra diğeri başladı gibi.

Bu güçlü form, **serializability** (sıralılaştırılabilirlik) olarak adlandırılır. Ancak çoğu zaman zayıf isolation level'lar kullanılır.

---

### 3.4 Durability (Dayanıklılık)

**Durability**, bir transaction başarıyla commit ettikten sonra, hardware hatası veya database çökmesi olsa bile yazılan tüm verinin kaybolmayacağının garantisidir.

Tek node'lu bir database'de durability genellikle verinin diske yazıldığı veya write-ahead log'a kaydedildiği anlamına gelir ve bu verinin bir çöküşten sonra kurtarılabileceği anlamına gelir.

Replikasyonlu bir database'de ise durability genellikle verinin başarılı commit raporu verilmeden önce birkaç node'a kopyalanması anlamına gelir.

**Mükemmel durability yok**: Bir teknoloji tek başına mutlak garantiler sağlayamaz. Çeşitli risk azaltma teknikleri vardır: diske yazma, uzak makinelere çoğaltma ve yedekleme. Bunlar birlikte kullanılmalıdır:

- SSD'ler güç kesilmesinde bazen veri güvencelerini ihlal edebilir; `fsync` bile her zaman doğru çalışmaz
- Bir çöküşten sonra dosyaların bozulmasına yol açabilecek storage engine ile filesystem uygulaması arasındaki ince etkileşimler
- SSD çalışmasının ilk dört yılında %30 ile %80'inin en az bir bad block geliştirdiği tespit edilmiştir
- Aşırı yıpranmış (çok yazma/silme döngüsü geçiren) bir SSD güç bağlantısı kesildiğinde sıcaklığa bağlı olarak haftalar ile aylar içinde veri kaybedebilir

## 4. Single-Object ve Multi-Object Operations

ACID bağlamında, atomicity ve isolation bir client'ın aynı transaction içinde birden fazla write yaptığında database'in ne yapması gerektiğini tanımlar:

- **Atomicity**: Write dizisinin ortasında bir hata oluşursa transaction abort edilmeli ve o ana kadar yapılan write'lar iptal edilmelidir. Kısaca database, tüm-ya-da-hiç (all-or-nothing) garantisi vererek sizi partial failure konusundaki endişeden kurtarır.
- **Isolation**: Eş zamanlı çalışan transaction'lar birbirini etkilememeli. Örneğin bir transaction birkaç write yapıyorsa, başka bir transaction bu write'ların ya tamamını ya da hiçbirini görmeli, bir alt kümesini değil.

Bu tanımlar birden fazla nesneyi (row, document, record) aynı anda değiştirmek istediğinizi varsayar. **Multi-object transaction**'lar çoğunlukla birbirleriyle senkronize tutulması gereken birden fazla veri parçası varsa gereklidir.

### E-posta Uygulaması Örneği

Klasik bir örnek: Bir kullanıcının okunmamış e-posta sayısını bir ayrı `unread_counter` alanında saklarsak (denormalization), yeni bir mesaj geldiğinde hem mesaj tablosunu hem de sayacı güncellememiz gerekir.

- **Isolation ihlali**: Kullanıcı, sayaç henüz güncellenmeden önce gelen e-postayı görürse sayacın sıfır olduğunu düşünebilir. Sayaç artışı henüz gerçekleşmemiş olduğundan "bir okunmamış mesaj var ama sayaç sıfır" gibi tutarsız bir görüntü oluşur.
- **Atomicity gereksinimi**: Sayaç güncellemesi başarısız olursa, e-posta ekleme işlemi de geri alınmalıdır.

```sql
-- Okunmamış mesaj sayısını sayan sorgu
SELECT COUNT(*) FROM emails WHERE recipient_id = 2 AND unread_flag = true
```

Relational database'lerde, aynı transaction içinde hangi read ve write operasyonlarının yer aldığı genellikle client'ın database sunucusuna olan TCP bağlantısına göre belirlenir. Belirli bir bağlantıda `BEGIN TRANSACTION` ile `COMMIT` arasındaki her şey aynı transaction'ın parçası olarak kabul edilir.

---

### 4.1 Single-Object Writes

Atomicity ve isolation, tek bir nesne değiştirilirken de geçerlidir. Örneğin database'e 20 KB'lık bir JSON belgesi yazıyorsunuz diyelim:

- İlk 10 KB gönderildikten sonra network bağlantısı kesilirse, database o ayrıştırılamaz 10 KB parçasını mı saklar?
- Database önceki değerin üzerine yazarken güç kesilirse, eski ve yeni değerlerin birbirine geçmiş hali mi kalır?
- Başka bir client yazma işlemi sırasında bu belgeyi okursa, kısmen güncellenmiş değeri mi görür?

Bu sonuçların her biri son derece kafa karıştırıcı olur. Bu nedenle storage engine'ler neredeyse evrensel olarak tek bir nesne üzerinde (örneğin bir key-value çifti) tek bir node üzerinde atomicity ve isolation sağlamayı hedefler:

- **Atomicity**: Crash recovery için log kullanılarak uygulanabilir (B-tree'leri güvenilir hale getirme ile ilgili)
- **Isolation**: Her nesne üzerinde lock kullanılarak uygulanabilir (herhangi bir anda yalnızca bir thread'in nesneye erişimine izin verilir)

Bazı database'ler daha karmaşık atomic operasyonlar da sağlar; örneğin bir **increment operasyonu** (read-modify-write döngüsüne ihtiyacı ortadan kaldırır) veya bir **conditional write** operasyonu (bir değer başkası tarafından eş zamanlı olarak değiştirilmemişse yazma yapılmasına izin verir). Bu operasyon, shared-memory concurrency'deki **compare-and-set (CAS)** veya **compare-and-swap** operasyonuna benzerdir.

Bu single-object operasyonlar faydalıdır çünkü birkaç client aynı nesneye eş zamanlı yazmaya çalışırken **lost update** sorununu önleyebilirler.

---

### 4.2 Multi-Object Transaction'lara Neden İhtiyaç Duyulur?

Çeşitli kullanım senaryolarında birden fazla nesneye yapılan write'ların koordine edilmesi gerekir:

1. **Relational data model**: Bir tablodaki bir satır, başka bir tablodaki satıra **foreign-key reference** içerebilir. Birbirine atıfta bulunan birden fazla kayıt eklenirken, foreign key'lerin doğru ve güncel olması gerekir.

2. **Document data model**: Document database'lerinde joins yerine denormalization teşvik edilir. Denormalize edilmiş bilginin güncellenmesi gerektiğinde birkaç belgeyi tek seferde güncellemek için transaction'lar oldukça faydalıdır.

3. **Secondary indexes**: Secondary index'e sahip database'lerde, bir değeri her değiştirdiğinizde index'in de güncellenmesi gerekir. Transaction izolasyonu olmadan, transaction isolation olmadan bir kaydın bir index'te görünmesi ama diğer index'te görünmemesi mümkündür.

---

### 4.3 Hata Yönetimi ve Abort'lar

Bir transaction'ın temel özelliği, hata oluşması durumunda abort edilip güvenle yeniden denenebilmesidir. ACID database'leri bu felsefe üzerine kuruludur: database, atomicity, isolation veya durability garantisini ihlal etme tehlikesiyle karşılaşırsa, transaction'ın yarı yarıya tamamlanmış halde kalmasına izin vermek yerine tamamen terk etmeyi tercih eder.

Ancak abort'tan sonra yeniden denemek mükemmel değildir:

- **Çift commit riski**: Transaction aslında başarılı oldu ancak server başarılı commit'i client'a bildirmeye çalışırken network kesildi; bu durumda yeniden deneme, transaction'ın iki kez gerçekleştirilmesine yol açabilir.
- **Overload feedback döngüsü**: Hata overload veya eş zamanlı transaction'lar arasındaki yüksek contention'dan kaynaklanıyorsa, yeniden denemek sorunu daha da kötüleştirebilir. Bunu önlemek için retry sayısını sınırlandırabilir ve **exponential backoff** kullanabilirsiniz.
- **Sadece geçici hatalar**: Yalnızca geçici hatalardan (örneğin deadlock, isolation ihlali, geçici network kesintisi) sonra yeniden denemek mantıklıdır. Kalıcı hatalardan (constraint ihlali gibi) sonra yeniden denemek anlamsızdır.
- **Yan etkiler**: Transaction'ın database dışında yan etkileri varsa, bu yan etkiler transaction abort edilse bile oluşabilir. Örneğin e-posta gönderiyorsanız, transaction'ı her yeniden denediğinizde e-postayı yeniden göndermek istemezsiniz.
- **Client crash**: İstemci yeniden deneme sırasında çökerse, database'e yazmaya çalıştığı veriler kaybolur.

## 5. Weak Isolation Levels (Zayıf İzolasyon Seviyeleri)

İki transaction aynı veriye erişmiyorsa ya da her ikisi de read-only ise, aralarında herhangi bir bağımlılık olmadığından güvenle parallel çalışabilirler. **Concurrency sorunları** (race condition'lar), ancak bir transaction başka bir transaction tarafından eş zamanlı olarak değiştirilen verileri okuduğunda ya da iki transaction aynı veriyi değiştirmeye çalıştığında ortaya çıkar.

Concurrency hataları, yalnızca zamanlamayla şanssız bir şekilde denk geldiğinizde tetiklendiğinden test yoluyla bulmak zordur. Bu tür zamanlama sorunları nadiren oluşur ve genellikle yeniden oluşturulması güçtür.

Bu nedenle database'ler, uzun süredir uygulama geliştiricilerinden concurrency sorunlarını gizlemeye **transaction isolation** sağlayarak çalışır. Teoride isolation, sanki hiçbir concurrency yokmuş gibi davranmanıza izin vermelidir. **Serializable isolation**, database'in transaction'ların aynı anda hiçbir concurrency olmaksızın sıralı olarak çalışmış gibi aynı etkiye sahip olacağını garanti ettiği anlamına gelir.

Pratikte isolation, maalesef bu kadar basit değildir. Serializable isolation'ın bir performans maliyeti vardır ve pek çok database bu bedeli ödemek istemez. Bu nedenle sistemler genellikle bazı concurrency sorunlarına karşı koruyan ama hepsine değil, **daha zayıf isolation seviyeleri** kullanır.

Zayıf transaction isolation ve race condition'lardan kaynaklanan concurrency hataları yalnızca teorik bir sorun değildir; gerçek hayatta ciddi para kayıplarına yol açmış, Bitcoin exchange'lerinin iflasına neden olmuş ve müşteri verilerinin bozulmasına sebep olmuştur.

---

### 5.1 Read Committed

**Read committed**, transaction isolation'ın en temel seviyesidir ve iki güvence sağlar:

1. **No dirty reads** (Kirli okumalar yok): Database'den okuma yaparken yalnızca commit edilmiş verileri görürsünüz. Başka bir transaction tarafından yazılmış ancak henüz commit edilmemiş verileri göremezsiniz.

2. **No dirty writes** (Kirli yazmalar yok): Database'e yazma yaparken yalnızca commit edilmiş verilerin üzerine yazarsınız. Başka bir transaction tarafından yazılmış ancak henüz commit edilmemiş bir verinin üzerine yazmazsınız.

#### Dirty Read Neden Kötüdür?

- Bir transaction birden fazla satırı güncellemesi gerekiyorsa, **dirty read**, başka bir transaction'ın güncellemelerin bir kısmını ama hepsini görmeyebileceği anlamına gelir. Bu, kafa karıştırıcı ve diğer transaction'ların yanlış kararlar almasına yol açan kısmen güncellenmiş bir durumu görmek demektir.
- Bir transaction abort edilirse, yaptığı write'ların geri alınması gerekir. Eğer database dirty read'lere izin verirse, bir transaction commit edilmemiş (sonunda geri alınacak olan) verileri görebilir. Bu da **cascading abort** olarak bilinen soruna yol açar.

#### Dirty Write Neden Kötüdür?

İki transaction aynı satırı güncellemeye çalışıyorsa, ikinci write'ın birincinin üzerine yazması normal bir durumdur. Ancak önceki write, henüz commit edilmemiş olan bir transaction'ın parçasıysa ve sonraki write bu commit edilmemiş değerin üzerine yazarsa buna **dirty write** denir.

Örnek: Bir online araba satış sitesinde Aaliyah ve Bryce aynı anda aynı arabayı satın almaya çalışıyor. İki database write gerekiyor: satışı yansıtmak için web sitesindeki listelemenin güncellenmesi ve alıcıya satış faturasının gönderilmesi. Dirty write ile satış Bryce'e (listing tablosunu kazanan update yapan kişi olarak) verilebilir ama fatura Aaliyah'a (invoices tablosunu kazanan update yapan kişi olarak) gönderilebilir. Read-committed isolation bu tür aksaklıkları önler.

#### Read Committed'in Uygulanması

Read committed, Oracle Database, PostgreSQL, SQL Server ve birçok başka database'de varsayılan isolation level'dır.

- **Dirty write önleme**: Database'ler genellikle **row-level lock** (satır düzeyinde kilit) kullanarak dirty write'ları önler. Bir transaction belirli bir satırı değiştirmek istediğinde, önce o satır üzerinde bir lock edinmesi gerekir. Transaction commit veya abort edene kadar lock'u tutması gerekir.

- **Dirty read önleme**: Write lock kullanımını gerektirmek mümkündür, ancak uzun süreli write transaction'lar pek çok read-only transaction'ı beklemek zorunda bırakabilir. Daha yaygın kullanılan yaklaşım: her write yapılan satır için database, hem eski commit edilmiş değeri hem de şu anda write lock'unu elinde bulunduran transaction tarafından belirlenen yeni değeri saklar. Transaction süresince, o satırı okuyan diğer transaction'lara yalnızca eski değer verilir. Yalnızca yeni değer commit edildiğinde transaction'lar yeni değeri okumaya geçer.

Bazı database'ler **read uncommitted** (okuma taahhüt edilmemiş) olarak adlandırılan daha da zayıf bir isolation level destekler. Bu seviye dirty write'ları önler ancak dirty read'leri önlemez.

---

### 5.2 Snapshot Isolation ve Repeatable Read

Read-committed isolation'a yüzeysel baktığınızda, bir transaction'ın ihtiyaç duyduğu her şeyi yaptığını düşünebilirsiniz: abort'lara izin verir, transaction'ların tamamlanmamış sonuçlarını okumayı önler ve eş zamanlı write'ların birbirine karışmasını önler. Ancak read-committed isolation kullanılırken bile pek çok concurrency hatası olabilir.

**Read skew** (okuma kayması): Örneğin Aaliyah'ın her biri 500 dolarlık iki hesapta toplam 1.000 dolar tasarrufu olsun. Bir transfer transaction'ı hesaplarından birinden diğerine 100 dolar aktarır. Aaliyah bu transaction işlenirken hesap bakiyelerini okursa, 500 dolarlık bakiyeyi (gelen ödeme henüz gelmeden önce) ve 400 dolarlık bakiyeyi (giden transfer yapıldıktan sonra) görme şansı olabilir. Aaliyah'a artık hesaplarında yalnızca 900 dolar varmış gibi görünebilir.

Bu sorun **non-repeatable read** olarak da adlandırılır ve genellikle şu durumlarda tolere edilemez:

- **Backups (Yedekleme)**: Büyük bir database için backup almak saatler sürebilir. Backup işlemi süresince write'lar database'e yapılmaya devam edecektir. Bu yüzden backup'ın bir kısmı daha eski veriler, diğer kısmı daha yeni veriler içerebilir. Böyle bir backup'tan restore yapmanız gerekirse, tutarsızlıklar (kaybolmuş para gibi) kalıcı hale gelir.
- **Analitik sorgular ve bütünlük kontrolleri**: Bazen database'in büyük bölümlerini tarayan bir sorgu çalıştırmak isteyebilirsiniz. Bu tür sorgular, üzerinde çalıştıkları veri sorgu yürütülürken değişiyorsa anlamsız sonuçlar döndürebilir.

#### Snapshot Isolation Nedir?

**Snapshot isolation**, bu soruna en yaygın çözümdür. Fikir şudur: her transaction, database'in bir **consistent snapshot** (tutarlı anlık görüntü) olarak okur — yani transaction başlangıcında database'de commit edilmiş tüm verileri görür. Veriler daha sonra başka bir transaction tarafından değiştirilse bile, her transaction yalnızca o belirli zaman noktasından eski verileri görür.

Snapshot isolation, backups ve analytics gibi uzun süreli, read-only sorgular için büyük bir avantajdır. PostgreSQL, MySQL (InnoDB storage engine ile), Oracle, SQL Server, TiDB ve diğerleri tarafından desteklenir. BigQuery gibi cloud veri ambarları da analitik sorgular için anlık görüntü görünümü sağladığından sıklıkla snapshot isolation kullanır.

---

### 5.3 Multiversion Concurrency Control (MVCC)

**MVCC**, snapshot isolation'ı uygulamak için database'lerin kullandığı genel bir tekniktir. Database, her satırın birden fazla commit edilmiş versiyonunu potansiyel olarak tutmalıdır, çünkü çeşitli devam eden transaction'lar, database'in durumunu farklı zaman noktalarında görmek zorunda kalabilir. Bir satırın birden fazla versiyonunu yan yana tuttuğu için bu teknik **multiversion concurrency control (MVCC)** olarak adlandırılır.

#### PostgreSQL'de MVCC Uygulaması

Bir transaction başladığında, ona benzersiz, her zaman artan bir **transaction ID (txid)** verilir. Bir transaction database'e bir şey yazdığında, yazdığı veriler yazarın transaction ID'siyle etiketlenir.

Tablodaki her satırın iki alanı bulunur:
- `inserted_by`: O satırı tabloya ekleyen transaction'ın ID'sini içerir
- `deleted_by`: Başlangıçta boştur. Bir transaction satırı silerse, satır database'den kaldırılmaz; bunun yerine, `deleted_by` alanı silme isteğinde bulunan transaction'ın ID'sine ayarlanarak silinmek üzere işaretlenir.

Bir **update** işlemi dahili olarak bir **delete** ve **insert** işlemine dönüştürülür.

#### Visibility Rules (Görünürlük Kuralları)

Bir transaction database'den okuduğunda, hangi satır versiyonlarının görünebileceğini ve hangilerinin görünmez olduğunu belirlemek için transaction ID'leri kullanılır:

1. Her transaction başlangıcında, database o sırada devam etmekte olan (henüz commit veya abort edilmemiş) tüm diğer transaction'ların bir listesini oluşturur. Bu transaction'ların yaptığı write'lar, sonradan commit etseler bile görmezden gelinir.
2. Daha sonraki bir transaction ID'sine sahip transaction'lar tarafından (yani mevcut transaction başladıktan sonra başlayan ve bu nedenle devam eden transaction'lar listesine dahil edilmeyen transaction'lar) yapılan write'lar, bu transaction'ların commit edip etmediğine bakılmaksızın görmezden gelinir.
3. Abort edilmiş transaction'lar tarafından yapılan write'lar görmezden gelinir.
4. Diğer tüm write'lar uygulamanın sorgularına görünürdür.

Başka bir deyişle, bir satır şu iki koşulun her ikisi doğruysa görünürdür:
- Okuyucunun transaction'ı başladığında, o satırı ekleyen transaction zaten commit edilmiş olmak zorundadır.
- Satır silinmek üzere işaretlenmemiş olmalı veya işaretlenmişse, silme isteğinde bulunan transaction okuyucunun transaction'ı başladığında henüz commit edilmemiş olmalıdır.

#### Indexes ve Snapshot Isolation

Multiversion bir database'de index'ler nasıl çalışır?

En yaygın yaklaşım: her index girişi, girişle eşleşen satırın versiyonlarından birine (en eskisi ya da en yenisi) işaret eder. Her satır versiyonu, bir sonraki en eski veya en yeni versyona bir referans içerebilir. Index kullanan bir sorgunun, görünür ve değerin sorgunun aradığıyla eşleştiği bir satır bulmak için satırlar üzerinde yinelenmesi gerekir.

CouchDB, Datomic ve LMDB gibi bazı database'ler, B-tree'lerin değiştirilemeyen (**immutable**) **copy-on-write** (yazarken kopyalama) varyantını kullanır. Bu yaklaşımda, güncellendiklerinde ağacın sayfalarının üzerine yazılmaz; bunun yerine, değiştirilen her sayfanın yeni bir kopyası oluşturulur.

---

### 5.4 İsimlendirme Karmaşası: Snapshot Isolation, Repeatable Read

Snapshot isolation farklı database'lerde farklı isimlerle adlandırılır:
- **PostgreSQL ve MySQL** → "repeatable read"
- **Oracle** → "serializable"

SQL standardının snapshot isolation kavramı yoktur, çünkü standart, snapshot isolation'ın henüz icat edilmediği 1975 tarihli System R'ın isolation level tanımına dayanmaktadır. Bunun yerine, snapshot isolation'a yüzeysel olarak benzer olan repeatable read isolation'ı tanımlar.

Sonuç olarak: kimse gerçekten repeatable read isolation'ın ne anlama geldiğini bilmemektedir.

## 6. Lost Updates (Kayıp Güncellemeler) Sorunu

**Lost update sorunu**: Bir uygulama database'den bir değer okur, bunu değiştirir ve değiştirilmiş değeri geri yazar (read-modify-write döngüsü). İki transaction bunu eş zamanlı olarak yaparsa, ikinci write ilk değişikliği dahil etmediğinden değişikliklerden biri kaybolabilir. İkinci write'ın birincinin üzerine yazmasına bazen ilk write'ı **"clobber"** (ezip geçmek) denir.

Bu durum çeşitli senaryolarda ortaya çıkar:
- Sayaç artırma veya hesap bakiyesi güncelleme (mevcut değeri okumayı, yeni değeri hesaplamayı ve güncellenmiş değeri geri yazmayı gerektirir)
- Karmaşık bir değerde yerel değişiklik yapma — örneğin bir JSON belgesi içindeki listeye eleman ekleme
- İki kullanıcının aynı anda bir wiki sayfasını düzenlemesi; her kullanıcı değişikliklerini, sayfanın tüm içeriğini gönderip database'de ne varsa üzerine yazarak kaydeder

### 6.1 Atomic Write Operations (Atomik Yazma Operasyonları)

Pek çok database, uygulama kodundaki read-modify-write döngülerine olan ihtiyacı ortadan kaldıran **atomic update operasyonları** sağlar. Bunlar genellikle mümkünse en iyi çözümdür:

```sql
-- Çoğu relational database'de concurrency-safe
UPDATE counters SET value = value + 1 WHERE key = 'foo';
```

MongoDB gibi document database'leri de bir JSON belgesinin bir bölümünde yerel değişiklikler yapmak için atomik operasyonlar sağlar. Redis, öncelik kuyruğu gibi veri yapılarını değiştirmek için atomik operasyonlar sağlar.

Atomic operasyonlar genellikle ilgili nesneye **exclusive lock** edinerek uygulanır, böylece transaction commit edilene kadar başka hiçbir transaction onu okuyamaz.

### 6.2 Explicit Locking (Açık Kilitleme)

Uygulamanın güncellemek istediği nesneleri açıkça kilitlemesi başka bir seçenektir. Örneğin:

```sql
BEGIN TRANSACTION;

SELECT * FROM figures
  WHERE name = 'robot' AND game_id = 222
  FOR UPDATE; -- Bu satır üzerindeki lock'u edinir

-- Verilen lock'la oynanan hamlenin geçerli olup olmadığını kontrol eden uygulama mantığı
UPDATE figures SET position = 'c4' WHERE id = 1234;

COMMIT;
```

`FOR UPDATE` ifadesi, database'e bu sorgu tarafından döndürülen tüm satırlar üzerinde bir lock edinmesini söyler.

### 6.3 Automatic Detection of Lost Updates (Otomatik Kayıp Güncelleme Tespiti)

Atomic operasyonlar ve lock'lar, lost update'leri önlemek için **pessimistic** (kötümser) yaklaşımlardır. **Snapshot isolation** uygulayan bazı database'ler, eş zamanlı transaction'ların detect edilmesine izin vererek bunu önler; transaction manager, bir lost update tespit ederse write transaction'larından birini abort eder.

Bu tekniğin büyük avantajı, database'in bunu verimli bir şekilde gerçekleştirebilmesi ve uygulamanın herhangi bir özel koduna ihtiyaç duymadan lost update'leri önleyebilmesidir.

PostgreSQL'in **repeatable read** (yani snapshot isolation), Oracle'ın **serializable** ve SQL Server'ın **snapshot isolation** seviyeleri lost update'leri otomatik olarak tespit eder. Bununla birlikte MySQL/InnoDB'nin repeatable read seviyesi lost update'leri tespit etmez.

### 6.4 Compare-and-Set (CAS)

Transactions sağlamayan database'lerde zaman zaman bulunan bir atomic operasyon olan **compare-and-set** operasyonu, bir write'ın yalnızca değer son okunduğundan bu yana değişmemişse gerçekleşmesine izin verir:

```sql
-- Şu anda beklediğiniz değerse güncelle:
UPDATE wiki_pages SET content = 'new content'
  WHERE id = 1234 AND content = 'old content';
```

Bu WHERE koşulu, "old content" içeriğini değiştirmeye çalışırsınız ancak bu içerik zaten değişmişse update herhangi bir satırla eşleşmez; bu nedenle update işleminin gerçekleşip gerçekleşmediğini kontrol etmeniz ve gerekirse yeniden denemeniz gerekir.

**Not**: Bu teknik, database'in snapshot'tan okumaya izin verip vermemesine bağlı olarak güvenli olmayabilir.

## 7. Write Skew ve Phantoms

**Write skew** (yazma kayması), lost update'den farklı ve daha ince bir race condition türüdür.

### Doktor Nöbet Örneği

Bir hastanede nöbet tutmakta olan doktorlar üzerine bir uygulama düşünün. En az bir doktorun nöbete gelmesi gerekiyor. Doktorlar nöbet değişimlerinden dışarı çıkmak isteyebilirler, bu nedenle bunu güvenli yapan bir transaction işlevi yazıyoruz:

```sql
BEGIN TRANSACTION;

-- Kaç doktorun şu anda nöbette olduğunu kontrol et
CURRENTLY_ON_CALL = (
  SELECT COUNT(*) FROM doctors
  WHERE on_call = TRUE
  AND shift_id = 1234
);

IF CURRENTLY_ON_CALL >= 2 THEN
  -- En az 2 doktor nöbette varsa nöbetten ayrılmak güvenlidir
  UPDATE doctors SET on_call = FALSE
  WHERE name = 'Aaliyah'
  AND shift_id = 1234;
END IF;

COMMIT;
```

İki doktorun her ikisi de eş zamanlı olarak nöbetten ayrılmak isterse ne olur? Her ikisi de aynı anda kontrol yapar ve 2 doktorun nöbette olduğunu görür. Her ikisi de on_call'ı false olarak günceller. Her iki transaction da commit eder. Artık nöbette kimse yok — bir kural ihlali!

Bu **write skew** örneğidir. Ne dirty write ne de lost update gibi görünür. Bunun yerine, iki eş zamanlı transaction aynı nesneleri okudu (doktor kayıtları) ama **farklı nesneleri** güncelledi (her transaction farklı doktoru güncelledi). Güncelleme, önceki sorgunun sonucunu geçersiz kılar.

### Write Skew'in Yapısı

Write skew birçok farklı write-write çakışma türünden biridir. Write skew'i önlemenin seçenekleri:

1. **Serializable isolation seviyesine geçmek** genellikle en temiz çözümdür.
2. **Explicitly locking** (SELECT FOR UPDATE) transaction'ın bağlı olduğu satırları kilitler:

```sql
BEGIN TRANSACTION;

SELECT * FROM doctors
  WHERE on_call = true
  AND shift_id = 1234 FOR UPDATE;

UPDATE doctors
  SET on_call = false
  WHERE name = 'Aaliyah'
  AND shift_id = 1234;

COMMIT;
```

### Write Skew'in Diğer Örnekleri

- **Meeting room booking**: Aynı toplantı odasının aynı saatte çift rezervasyon yapılması
- **Multiplayer game**: İki oyuncunun farklı figürleri tahta üzerindeki aynı konuma taşıması
- **Username claim**: İki kullanıcının aynı anda aynı kullanıcı adıyla hesap oluşturmaya çalışması
- **Double-spending önleme**: İki kullanıcı harcama kalemlerini eş zamanlı ekleyerek bakiyeyi negatife düşürme riski

---

### 7.1 Phantoms Causing Write Skew (Hayalet Kayıtların Write Skew'e Yol Açması)

Önceki örneklerin hepsi benzer bir deseni takip eder:

1. Bir `SELECT` sorgusu, bir arama koşulunu karşılayan satırları bularak bir gereksinimin sağlanıp sağlanmadığını kontrol eder.
2. Birinci sorgunun sonucuna göre uygulama kodunun nasıl devam edeceğine karar verir.
3. Uygulama devam etmeye karar verirse database'e bir write (`INSERT`, `UPDATE` veya `DELETE`) yapar ve transaction'ı commit eder. Bu write'ın etkisi, adım 2'deki kararın ön koşulunu değiştirir.

**Phantom** (hayalet): Bir transaction'daki write'ın başka bir transaction'daki arama sorgusunun sonucunu değiştirdiği duruma denir.

**Snapshot isolation**, read-only sorgulardaki phantom'ları önler, ancak read/write transaction'larında phantom'lar, write skew'in özellikle zorlu durumlarına yol açabilir.

#### Materializing Conflicts (Çakışmaları Somutlaştırma)

Phantom'ların sorunu lock'ları ekleyebileceğimiz hiçbir nesnenin bulunmamasıdır — belki database'e yapay olarak bir lock nesnesi ekleyebiliriz?

Örneğin meeting room reservation durumunda zaman dilimlerini ve odaları içeren bir tablo oluşturabilir ve ilerleyen altı ay için tüm olası oda-zaman dilimi kombinasyonları için satırlar oluşturabilirsiniz.

Bir rezervasyon oluşturmak isteyen bir transaction, tablodaki istenen oda ve zaman dilimine karşılık gelen satırlara lock ekleyebilir (`SELECT FOR UPDATE`). Lock'ları edindikten sonra transaction, örtüşen rezervasyonları kontrol edebilir ve her zamanki gibi yeni bir rezervasyon ekleyebilir.

Bu yaklaşıma **materializing conflicts** (çakışmaları somutlaştırma) adı verilir çünkü bir phantom alır ve database'de var olan somut bir satır kümesi üzerindeki lock çakışmasına dönüştürür. Ancak bu yaklaşım hatalara yatkın ve çirkindir. Çoğu durumda **serializable isolation** kullanmak tercih edilir.

## 8. Serializability (Sıralılaştırılabilirlik)

Bu ana kadar transaction'ların race condition'lara açık olduğu çeşitli örnekler gördük. Bazı race condition'lar read-committed ve snapshot isolation seviyeleri tarafından önlenir, ancak hepsi değil — özellikle write skew ve phantom'larla ilgili özellikle zorlu örneklerle karşılaştık:

- Isolation level'lar anlaşılması güçtür ve farklı database'lerde tutarsız biçimde uygulanır
- Uygulama koduna bakarak belirli bir isolation level'da güvenli olup olmadığını söylemek zor olabilir
- Race condition'ları tespit etmemize yardımcı olacak iyi araçlar yoktur

Bu yeni bir sorun değil. 1970'lerden beri böyle. Araştırmacıların cevabı basit: **serializable isolation kullanın!**

**Serializable isolation**, en güçlü isolation seviyesidir. Transaction'lar parallel yürütülse bile, nihai sonucun hiçbir concurrency olmaksızın sırayla (serially), birer birer yürütülmüş gibi aynı olacağını garanti eder. Böylece, transaction'lar bireysel olarak çalıştırıldığında doğru davranıyorlarsa, eş zamanlı çalıştırıldıklarında da doğru davranmaya devam ederler — başka bir deyişle, database tüm olası race condition'ları önler.

Serializability'yi uygulamak için üç ana teknik vardır:

1. **Actual Serial Execution** (Gerçek Sıralı Yürütme)
2. **Two-Phase Locking (2PL)**
3. **Serializable Snapshot Isolation (SSI)**

---

### 8.1 Actual Serial Execution (Gerçek Sıralı Yürütme)

Concurrency sorunlarını önlemenin en basit yolu concurrency'yi tamamen ortadan kaldırmaktır: tek seferde yalnızca bir transaction yürütmek, sırayla, tek thread üzerinde. Bunu yaparak transaction'lar arasındaki çakışmaları tespit etme ve önleme sorununu tamamen bertaraf ederiz; elde edilen isolation tanım gereği serializabledır.

Bu bariz bir fikir gibi görünse de, single-threaded bir döngünün uygulanabilir olduğuna ancak 2000'lerde karar verildi. Bu değişimi mümkün kılan iki gelişme:

1. **RAM fiyatlarının düşmesi**: Artık pek çok kullanım senaryosu için tüm aktif veri kümesini memory'de tutmak mümkün. Bir transaction'ın erişmesi gereken tüm veriler memory'de olduğunda, diskten yüklenmesini beklemek gerekmediğinden transaction'lar çok daha hızlı yürütülebilir.

2. **OLTP transaction'larının kısa olduğu fark edildi**: OLTP transaction'larının genellikle kısa olduğu ve yalnızca az sayıda read ve write yaptığı anlaşıldı. Buna karşılık uzun süreli analitik sorgular genellikle read-only'dir, bu nedenle sıralı yürütme döngüsünün dışında consistent snapshot'ta çalıştırılabilirler.

Bu yaklaşım **VoltDB/H-Store**, **Redis** ve **Datomic** gibi database'lerde uygulanmaktadır.

#### Stored Procedures

Single-threaded serial execution kullanan sistemler interactive multi-statement transaction'lara izin vermez. Bunun yerine uygulama:
- Transaction'ı tek bir statement ile sınırlandırmalıdır ya da
- Transaction kodunun tamamını önceden **stored procedure** olarak database'e göndermelidir.

Stored procedure'ların gerektirdiği tüm veriler memory'deyse, network veya disk I/O beklemeden çok hızlı yürütülebilir.

**Stored procedure'ların dezavantajları:**
- Her database vendor'ının stored procedure için kendi dili var (Oracle PL/SQL, SQL Server T-SQL, PostgreSQL PL/pgSQL)
- Database'de çalışan kodu yönetmek zordur; debug etmek, version control'e entegre etmek ve test etmek daha güç
- Database, uygulama sunucusundan genellikle çok daha performans hassastır; kötü yazılmış stored procedure çok daha büyük sorunlara neden olabilir

**Modern stored procedure'lar**: Bu sorunların üstesinden gelinebilir. VoltDB Java veya Groovy kullanır, Datomic Java veya Clojure kullanır, Redis Lua kullanır, MongoDB JavaScript kullanır.

#### Sharding (Parçalama)

Single-threaded transaction execution'ın throughput'u tek bir CPU core ile sınırlıdır. Tek thread'den en iyi şekilde yararlanmak için transaction'ların geleneksel biçimlerinden farklı yapılandırılması gerekir.

Eğer veriler birden fazla shard arasında bölünebiliyorsa, farklı shardlar üzerindeki transaction'ları parallel yürütmek için her biri tek bir thread'e sahip birden fazla shard kullanmak mümkündür. Her transaction yalnızca tek bir shard içindeki veriye erişmek zorundaysa, shard'lar birbirinden bağımsız olarak yürütebilir ve throughput CPU core sayısıyla doğrusal olarak ölçeklenebilir.

Birden fazla shard'ı kapsayan transaction'lar mümkündür ancak önemli ölçüde ek yük getirir ve throughput'u azaltır.

---

### 8.2 Two-Phase Locking (2PL)

Yaklaşık 30 yıl boyunca, serializability uygulamak için yalnızca bir yaygın algoritma mevcuttu: **two-phase locking (2PL)** (iki aşamalı kilitleme).

**Not**: 2PL (two-phase locking), transaction commit protokolü olarak kullandığımız 2PC (two-phase commit) ile karıştırılmamalıdır.

**2PL ile birden fazla transaction birbirine bir çakışma olmaksızın aynı nesneyi okuyabilir. Ancak aşağıdakiler için:**

- Bir nesneyi **yazmak** istiyorsanız, başka hiçbir transaction'ın o nesneyi okumaması veya yazmaması gerekir
- Bir nesneyi **okumak** istiyorsanız, başka hiçbir transaction'ın o nesneyi yazmaması gerekir (ancak birden fazla transaction aynı anda okuyabilir)

2PL'de, **writers** (yazıcılar) sadece diğer writer'ları değil, **reader'ları** da engeller.

#### 2PL'nin Uygulanması

2PL, database'deki her nesne üzerinde bir lock edinilerek uygulanır. Lock ya **shared mode** (paylaşımlı mod) ya da **exclusive mode** (özel mod) olabilir.

- Eğer bir transaction bir nesneyi okumak istiyorsa, önce **shared mode**'da lock edinmelidir. Birden fazla transaction aynı anda shared lock'u tutabilir, ama nesne üzerinde exclusive lock varsa, transaction wait etmek zorunda kalır.
- Eğer bir transaction bir nesneyi yazmak istiyorsa, önce **exclusive mode**'da lock edinmelidir. Nesne üzerinde shared ya da exclusive lock varsa, transaction wait etmek zorundadır.
- Önce okuyan sonra yazan bir transaction shared lock'ı exclusive lock'a **upgrade** etmek zorundadır.
- Transaction lock edindikten sonra, transaction commit veya abort edene kadar lock'u tutmalıdır.

**Two-phase** kısmı şu anlama gelir: İlk fazda (transaction yürütülürken) lock'lar edinilir; ikinci fazda (transaction bittikten sonra) lock'lar serbest bırakılır.

#### Deadlock'lar

2PL kullanılırken database'lerin lock bekleme döngüsü oluşmasını (A, B'yi bekliyor; B, A'yı bekliyor) tespit edip bunu sonlandırarak **deadlock**'larla başa çıkmaları gerekir. Database, transaction'lardan birini otomatik olarak abort ederek deadlock'ı kırar. Abort edilen transaction, uygulama tarafından yeniden denenebilir.

#### 2PL'nin Performansı

2PL'de transaction throughput'u ve sorgulara verilen yanıt süresi, lock contention nedeniyle lock olmayan sistemlere kıyasla önemli ölçüde daha kötüdür. 2PL kullanırken sıkça deadlock oluşabilir; bu da transaction'ların abort edilmesi ve yeniden denenmesine neden olur.

#### Predicate Locks ve Index-Range Locks

**Predicate lock** (tahmin kilidi), bir transaction'ın eriştiği tüm nesnelere değil, belirli bir arama koşulunu karşılayan tüm nesnelere ait bir kilide benzer. 2PL, phantom'lara karşı koruma sağlamak için predicate lock kullanır. Mevcut nesneler üzerindeki lock'ların yanı sıra, future INSERT veya UPDATE'in kilitli bir sorguyla eşleşmesi durumunda diğer transaction'ların o nesneye write yapmasını engeller.

**Index-range lock** (indeks aralığı kilidi): Predicate lock pratikte iyi performans göstermez çünkü çok sayıda aktif lock varsa, eşleşen lock'ları kontrol etmek zaman alır. Database'lerin çoğu predicate lock uygulamaz; bunun yerine, daha az hassas olmakla birlikte çok daha düşük overhead'e sahip olan **index-range lock**'ları (next-key locking olarak da bilinir) kullanır.

```sql
-- Index-range lock örneği:
-- room_id index'i üzerinde lock, 123 numaralı odanın tüm zamanlarını kapsar
SELECT * FROM bookings WHERE room_id = 123 FOR UPDATE;
```

---

### 8.3 Serializable Snapshot Isolation (SSI)

Bir tarafta iyi performans göstermeyen (2PL) veya iyi ölçeklenmeyen (serial execution) serializability uygulamaları var. Öte yanda iyi performans gösteren ancak çeşitli race condition'lara yatkın weak isolation level'lar var. Serializable isolation ve iyi performans temelden birbirine zıt mı?

Görünüşe göre hayır: **Serializable Snapshot Isolation (SSI)** adlı bir algoritma, snapshot isolation'a kıyasla yalnızca küçük bir performans cezasıyla tam serializability sağlar. SSI nispeten yeni bir yaklaşımdır; 2008'de ilk kez tanımlanmıştır.

Bugün SSI ve benzeri algoritmalar şu sistemlerde kullanılmaktadır: PostgreSQL (serializable isolation level), SQL Server'ın In-Memory OLTP/Hekaton, CockroachDB, FoundationDB ve BadgerDB gibi embedded storage engine'leri.

#### Pessimistic vs Optimistic Concurrency Control

**2PL pesimist (kötümser) bir concurrency control mekanizmasıdır**: Bir şeyin yanlış gidebileceği ihtimali olduğunda (başka bir transaction tarafından tutulan lock ile belirtildiği gibi), tekrar harekete geçmeden önce durumun yeniden güvenli hale gelmesini beklemek daha iyidir.

**Sıralı yürütme**, bir anlamda aşırı pesimisttir; her transaction'ın tüm database üzerinde exclusive lock tutmasına eşdeğerdir.

Buna karşılık, **Serializable Snapshot Isolation bir optimist (iyimser) concurrency control tekniğidir**. Optimist, bu bağlamda şu anlama gelir: Potansiyel olarak tehlikeli bir şey olursa engellemek yerine, transaction'lar her şeyin yoluna gireceği umuduyla devam eder. Bir transaction commit etmek istediğinde database, kötü bir şeyin olup olmadığını kontrol eder; olduysa transaction abort edilir ve yeniden denenmek zorunda kalır. Yalnızca sıralı olarak yürütülen transaction'ların commit etmesine izin verilir.

**Optimistic concurrency control** eski bir fikirdir ve avantajları ile dezavantajları uzun süredir tartışılmaktadır. Yüksek contention durumunda (birçok transaction aynı nesnelere erişmeye çalışıyorsa) kötü performans gösterir. Ancak spare kapasitesi yeterliyse ve transaction'lar arasındaki contention fazla değilse, optimistic concurrency control teknikleri genellikle pessimistic tekniklere göre daha iyi performans gösterir.

#### SSI'nın Çalışma Prensibi

SSI, snapshot isolation üzerine inşa edilir. Snapshot isolation'a ek olarak SSI, okuma ve yazma işlemleri arasındaki serializability çakışmalarını tespit etmek ve hangi transaction'ların abort edileceğini belirlemek için bir algoritma ekler.

SSI'nın write skew ve phantom'larla başa çıkması için gerekli bilgi: bir transaction, snapshot'tan okunan veriye dayanarak bir action alıyorsa, o snapshot'ın zamanından bu yana okunan verinin değişmiş olup olmadığını tespit etmesi gerekir.

SSI, iki durumu kontrol eder:

1. **Stale MVCC okumalarının tespiti** (okumadan önce gerçekleşen uncommitted write): Bir transaction, snapshot'tan okurken başka bir transaction'ın taahhüt edilmemiş yazma işlemlerini görmezden gelir. Ancak bu transaction commit etmek istediğinde, görmezden gelinen write'ların herhangi biri commit edilmiş mi diye kontrol eder. Commit edilmişse, transaction abort edilmelidir.

2. **Önceki okumaları etkileyen write'ların tespiti** (okumadan sonra gerçekleşen write): Index-range lock'larına benzer bir teknik kullanılır, ancak SSI lock'ları diğer transaction'ları engellemez. Bir transaction database'e yazdığında, son zamanlarda etkilenen verileri okuyan diğer transaction'ları index'lerde aramalıdır.

#### SSI'nın Performansı

2PL'ye kıyasla SSI'nın en büyük avantajı: bir transaction, başka bir transaction tarafından tutulan lock'u beklemek zorunda değildir. Snapshot isolation'da olduğu gibi writer'lar reader'ları engellemez ve bunun tersi de geçerlidir. Bu tasarım prensibi, sorgu gecikmesini çok daha tahmin edilebilir ve değişken olmayan hale getirir.

Sıralı yürütmeyle karşılaştırıldığında, SSI tek bir CPU core'un throughput'uyla sınırlı değildir. FoundationDB, serialization çakışması tespitini birden fazla makineye dağıtarak çok yüksek throughput'a ölçeklenmesine izin verir.

SSI'nın genel performansı üzerindeki abort oranının etkisi önemlidir. Uzun bir süre boyunca veri okuyup yazan bir transaction, büyük olasılıkla çakışmalarla karşılaşır ve abort edilir.

## 9. Distributed Transactions (Dağıtık İşlemler)

**Distributed transaction** (dağıtık işlem): Transaction, birden fazla node'u kapsayan yazma işlemi içeriyorsa ortaya çıkar — örneğin sharded database'in birden fazla shardına dokunması gereken veya bir global secondary index'i güncellemesi gereken bir transaction.

Distributed transaction'larda concurrency control algoritmaları, single-node'lu olanlarla genel olarak aynıdır; 2PL dağıtık bir ortamda çalışır, SSI için ise distributed serializability checker'lar mevcuttur.

### Atomicity Sorunu Distributed Transaction'larda

Distributed transaction'larda atomicity sağlamak tamamen yeni bir zorluğu beraberinde getirir.

Single-node transaction'larda atomicity, storage engine tarafından sağlanır. Database düğümü, transaction'ın write'larını kalıcı hale getirir ve ardından commit kaydını log'a ekler.

Bir distributed transaction'da, commit isteğini tüm node'lara bağımsız olarak göndermek yeterli değildir. Commit'in bazı node'larda başarılı, bazılarında başarısız olması kolayca mümkündür:

- Bazı node'lar bir constraint ihlali veya çakışma tespit edebilir ve abort gerektirebilir
- Commit isteklerinin bir kısmı timeout nedeniyle abort edilebilirken diğerleri iletilir
- Bazı node'lar commit kaydı tamamen yazılmadan önce çöküp kurtarma sırasında transaction'ı geri alabilir

Bazı node'lar transaction'ı commit edip diğerleri abort ederse, node'lar birbirleriyle tutarsız hale gelir. Ve bir transaction bir node'da commit edildikten sonra başka bir node'da abort edildiği ortaya çıksa bile geri alınamaz — çünkü veriler commit edildiğinde read-committed veya daha güçlü isolation altında diğer transaction'lara görünür hale gelir.

Daha iyi bir yaklaşım, transaction'a dahil olan node'ların hepsinin commit etmesini veya hepsinin abort etmesini sağlamak ve ikisinin karışımını önlemektir. Bunu başarmak **atomic commitment problemi** olarak bilinir.

---

### 9.1 Two-Phase Commit (2PC)

**Two-phase commit (2PC)**, birden fazla node üzerinde atomic transaction commit'i sağlamak için kullanılan bir algoritmadır. Bazı database'lerde dahili olarak kullanılır, bazılarında ise **XA transactions** (Java Transaction API aracılığıyla desteklenen) veya SOAP web servisleri için WS-AtomicTransaction şeklinde uygulamalara sunulur.

#### 2PC'nin Yapısı

Single-node transaction'dan farklı olarak, 2PC'de commit/abort süreci iki fazdan oluşur (bu nedenle "two-phase" adı verilir).

2PC, single-node transaction'larda normalde görünmeyen yeni bir bileşen kullanır: **coordinator** (koordinatör) ya da **transaction manager** (işlem yöneticisi). Koordinatör genellikle transaction isteğinde bulunan uygulama süreciyle aynı süreçte bir kütüphane olarak uygulanır. Koordinatörlere örnek olarak Narayana, JOTM, BTM ve MSDTC verilebilir.

2PC kullanıldığında, distributed transaction, uygulamanın birden fazla database node'unda veri okuması ve yazmasıyla başlar. Bu database node'larına transaction'daki **participants** (katılımcılar) denir. Uygulama commit etmeye hazır olduğunda, koordinatör birinci faza (1. faz) başlar:

**Faz 1 (Hazırlık):**
- Koordinatör, her node'a commit edip edemeyeceğini soran bir **prepare** isteği gönderir
- Koordinatör, participant'ların yanıtlarını takip eder

**Faz 2 (Commit/Abort):**
- Tüm participant'lar evet yanıtı verirse, koordinatör 2. fazda bir **commit** isteği gönderir
- Herhangi bir participant hayır yanıtı verirse, koordinatör 2. fazda tüm node'lara bir **abort** isteği gönderir

Bu süreç, Batı kültürlerindeki geleneksel evlilik törenine benzetilebilir: evlendirici, her ortağa ayrı ayrı diğeriyle evlenmek isteyip istemediğini sorar ve genellikle her ikisinden de "evet" yanıtını alır. Her iki onayı aldıktan sonra evlendirici çifti evli ilan eder ve transaction commit edilmiş olur.

#### 2PC'yi Garantileyen Sistem: Söz Sistemi

2PC'nin neden atomicity sağladığını anlamak için süreci biraz daha ayrıntılı ele almak gerekir:

1. Uygulama distributed transaction başlatmak istediğinde, koordinatörden bir transaction ID ister. Bu transaction ID **globally unique** (küresel olarak benzersiz) dir.

2. Uygulama, participant'ların her birinde single-node transaction başlatır ve globally unique transaction ID'yi bu single-node transaction'a ekler. Tüm read'ler ve write'lar bu single-node transaction'larından birinde gerçekleştirilir.

3. Uygulama commit etmeye hazır olduğunda, koordinatör global transaction ID ile etiketlenmiş tüm participant'lara bir prepare isteği gönderir. Bu isteklerden biri başarısız olursa veya zaman aşımına uğrarsa, koordinatör o transaction ID için tüm participant'lara bir abort isteği gönderir.

4. Bir participant prepare isteği aldığında, transaction'ı her koşul altında kesinlikle commit edebileceğinden emin olur. Bu, tüm transaction verilerini diske yazmayı ve çakışma veya constraint ihlallerini kontrol etmeyi içerir. Koordinatöre evet yanıtı vererek, node aslında commit etmeden transaction'ı hata olmaksızın commit etmeyi **söz verir**. Başka bir deyişle, participant transaction'ı abort etme hakkından vazgeçer, ancak transaction'ı gerçekten commit etmez.

5. Koordinatör, tüm prepare isteklerine yanıtları aldığında, transaction'ı commit mi yoksa abort mu edeceğine kesin bir karar verir (yalnızca tüm participant'lar evet ise commit eder). Koordinatör, daha sonra çökmesi durumunda hangi yönde karar verdiğini bilmesi için bu kararı **transaction log**'una yazmalıdır. Buna **commit point** (commit noktası) denir.

6. Koordinatörün kararı diske yazıldıktan sonra, commit veya abort isteği tüm participant'lara gönderilir. Bu istek başarısız olursa veya zaman aşımına uğrarsa, koordinatör başarılı olana kadar sonsuza kadar yeniden denemelidir. Artık geri dönüş yok; karar commit etmekse, bu karar kaç yeniden deneme gerektirirse gerektirsin uygulanmalıdır.

**Protokol iki kritik "geri dönüş noktası" içerir**:
1. Bir participant evet oyu verdiğinde, daha sonra kesinlikle commit edebileceğini söz verir
2. Koordinatör karar verdiğinde, bu karar geri alınamaz

#### Coordinator Failure (Koordinatör Arızası)

Peki koordinatör başarısız olursa ne olur? Prepare isteği gönderilmeden önce başarısız olursa, participant transaction'ı güvenle abort edebilir. Ancak her participant evet oyu verdikten sonra, commit ya da abort kararı gelene kadar bir participant ne yapacağını bilemez — buna **in doubt** (şüphe içinde) veya **uncertain** (belirsiz) denir.

Teoride, koordinatör çöküp yeniden başlatılırsa, durumunu log'dan kurtarmalı ve in-doubt transaction'ları çözmelidir.

#### Locking During In-Doubt Period (Şüphe Süresinde Kilitleme)

In-doubt transaction'lar neden bu kadar önemli? Sistem neden bu transaction'ı görmezden gelemez?

**Locking** sorunu! Database transaction'ları, dirty write'ları önlemek için değiştirdikleri satırlar üzerinde row-level exclusive lock edinir. 2PC kullanırken, bir transaction commit veya abort edene kadar lock'ları tutmak zorundadır. Dolayısıyla, koordinatör çöküp yeniden başlatılması 20 dakika alırsa, lock'lar 20 dakika tutulur. Bu süre zarfında başka hiçbir transaction aynı satırları değiştiremez.

Lock'lar tutulurken, uygulamanızın büyük bir kısmı in-doubt transaction çözülene kadar kullanılamaz hale gelebilir.

---

### 9.2 XA Transactions

**X/Open XA** (eXtended Architecture), heterojen teknolojiler genelinde 2PC uygulamak için bir standarttır. 1991'de tanıtılmıştır ve yaygın biçimde uygulanmıştır. XA, PostgreSQL, MySQL, Db2, SQL Server ve Oracle dahil pek çok geleneksel relational database ve ActiveMQ, HornetQ, MSMQ ve IBM MQ dahil message broker'lar tarafından desteklenir.

XA bir network protokolü değildir — sadece bir transaction koordinatörüyle arayüz oluşturmak için bir C API'sidir. Java EE uygulamalarında XA transaction'ları, JDBC ve JMS API'lerini kullanan driver'ların desteklediği Java Transaction API (JTA) kullanılarak uygulanır.

#### XA Transaction'larının Sorunları

1. **Single point of failure (Tek başarısızlık noktası)**: Single-node koordinatör, tüm sistem için tek başarısızlık noktasıdır

2. **Temel bir sorun**: XA, koordinatör ile transaction katılımcıları arasında doğrudan iletişim yolu sağlamaz; yalnızca uygulama kodu aracılığıyla iletişim kurabilirler

3. **Lowest common denominator tuzağı**: XA, çok çeşitli sistemlerle uyumlu olmak zorunda olduğundan, en küçük ortak paydaya dayanmak zorundadır

4. **Deadlock detection**: Farklı sistemler arasındaki deadlock'ları tespit edemez; bunun için standartlaştırılmış bir protokol gerekirdi

5. **SSI ile uyumsuz**: XA, SSI ile çalışmaz çünkü bunun için farklı sistemlerde çakışmaları belirleyecek bir protokol gerekir

**Heuristic decisions (Sezgisel kararlar)**: Pek çok XA uygulaması, bir participant'ın koordinatörden kesin bir karar olmaksızın tek taraflı olarak in-doubt transaction'ı abort veya commit etmeye karar verebileceği acil durum kaçış kapısı sunar. Burada **heuristic**, **"muhtemelen atomicity'yi ihlal etmek"** için kullanılan bir örtmecedir. Sezgisel kararlar yalnızca felaket niteliğindeki durumlardan çıkmak için tasarlanmıştır, rutin kullanım için değil.

---

### 9.3 Database-Internal Distributed Transactions

Heterojen storage teknolojilerini kapsayan distributed transaction'lar ile bir sisteme dahili olanlar arasında büyük bir fark vardır — yani, tüm katılımcı node'larının aynı yazılımı çalıştıran aynı database'in parçası olduğu durumlar.

Bu tür **database-internal distributed transaction**'lar, NewSQL database'lerinin tanımlayıcı özelliğidir: **CockroachDB**, **TiDB**, **Spanner**, **FoundationDB** ve **YugabyteDB**. **Kafka** gibi bazı message broker'lar da dahili distributed transaction'ları destekler.

Bu sistemlerin çoğu, birden fazla sharda yazma işlemlerinin atomicity'sini sağlamak için 2PC kullanır, ancak XA transaction'larıyla aynı sorunlarla karşılaşmazlar. Distributed transaction'ları herhangi bir başka teknoloji ile arayüz oluşturmak zorunda olmadığından, lowest-common-denominator tuzağından kaçınırlar — bu sistemlerin tasarımcıları daha güvenilir ve daha hızlı olan daha iyi protokoller kullanmakta özgürdür.

XA'nın en büyük sorunları şu yollarla çözülebilir:
- Primary koordinatör çökerse, başka bir koordinatör node'una otomatik failover ile koordinatörü çoğaltma
- Koordinatör ve data shard'larının aracı uygulama kodu olmadan doğrudan iletişim kurmasına izin verme
- Bir shard'daki hata nedeniyle transaction'ın abort edilmesi riskini azaltmak için participant shard'ları çoğaltma
- Deadlock tespitini ve shardlar arasında tutarlı okuma işlemlerini destekleyen bir distributed concurrency control protokolüyle atomik commitment protokolünü birleştirme

---

### 9.4 Exactly-Once Message Processing

Distributed transaction'lar için önemli bir kullanım senaryosu, bir işlem sırasında çökme olsa ve işlemenin yeniden denenmesi gerekse bile bir operasyonun tam olarak bir kez gerçekleşmesini sağlamaktır.

Bir message broker ve database üzerinde atomic olarak commit edebilirseniz, database write'larından kaynaklanan işleme başarıyla commit edilmişse ve yalnızca o zaman mesajı broker'a acknowledge edebilirsiniz.

**Ancak, exactly-once semantics için distributed transaction'lara ihtiyaç yoktur!** Yalnızca database içindeki transaction'lara ihtiyaç vardır:

1. Her mesajın benzersiz bir ID'si olduğunu varsayın ve database'de işlenen mesaj ID'lerinin bir tablosu bulundurun. Broker'dan bir mesaj işlemeye başladığınızda, database'de yeni bir transaction başlatın ve mesaj ID'sini kontrol edin. Aynı mesaj ID'si database'de zaten mevcutsa, zaten işlendiğini bilirsiniz.

2. Mesaj ID'si database'de yoksa, onu tabloya ekleyin. Ardından mesajı işleyin ve aynı transaction içinde database'e ek yazma işlemleri gerçekleştirebilirsiniz.

3. Database transaction başarıyla commit edildiğinde, mesajı broker'a acknowledge edin.

4. Mesaj başarıyla broker'a acknowledge edildikten sonra, aynı mesajı tekrar işlemeyeceğini bilirsiniz; mesaj ID'sini database'den silebilirsiniz (ayrı bir transaction içinde).

Mesaj ID'sini database'de kaydetmek, mesaj işlemeyi **idempotent** hale getirir; böylece mesaj işleme yan etkilerini çoğaltmadan güvenle yeniden denenebilir. Kafka Streams gibi stream processing framework'leri exactly-once semantics elde etmek için benzer bir yaklaşım kullanır.

## 10. Özet: İzolasyon Seviyeleri Karşılaştırması

| Isolation Level | Dirty Read | Dirty Write | Read Skew | Lost Update | Write Skew | Phantom |
|-----------------|:---:|:---:|:---:|:---:|:---:|:---:|
| **Read Uncommitted** | ✓ | ✗ | ✓ | ✓ | ✓ | ✓ |
| **Read Committed** | ✗ | ✗ | ✓ | ✓ | ✓ | ✓ |
| **Snapshot Isolation** | ✗ | ✗ | ✗ | ✗* | ✓ | ✓** |
| **Serializable (2PL/SSI)** | ✗ | ✗ | ✗ | ✗ | ✗ | ✗ |

*✗ = Önlenir, ✓ = Oluşabilir*  
*\* Snapshot isolation bazı lost update durumlarını önler (bazı veritabanları otomatik tespit eder)  
*\*\* Snapshot isolation read-only sorgulardaki phantom'ları önler, ancak read/write transaction'larında önlemez

---

## 11. Serializability Yaklaşımlarının Karşılaştırması

| Yaklaşım | Avantajlar | Dezavantajlar |
|----------|-----------|---------------|
| **Actual Serial Execution** | Basit, deadlock yok, yüksek throughput (single-thread) | Tek CPU core ile sınırlı, stored procedure gerektiriyor, tek shard ile sınırlı |
| **Two-Phase Locking (2PL)** | Güçlü garantiler | Deadlock riski, writer'lar reader'ları bloklar, yüksek latency |
| **Serializable Snapshot Isolation (SSI)** | Writer'lar reader'ları bloklamaz, iyi concurrency, dağıtık sistemlere uygun | Yüksek contention'da abort oranı artar, ek overhead |

---

## 12. Terimler Sözlüğü

| İngilizce Terim | Açıklama |
|-----------------|----------|
| **Transaction** | Birden fazla read/write'ı tek bir logical unit'e gruplandıran mekanizma |
| **Commit** | Transaction'ın başarıyla tamamlanıp değişikliklerin kalıcı hale gelmesi |
| **Abort / Rollback** | Transaction'ın geri alınması ve tüm değişikliklerin iptali |
| **ACID** | Atomicity, Consistency, Isolation, Durability |
| **BASE** | Basically Available, Soft state, Eventual consistency |
| **Dirty Read** | Henüz commit edilmemiş verinin okunması |
| **Dirty Write** | Commit edilmemiş bir write'ın üzerine yazma |
| **Read Skew** | Tutarsız anlık görüntüden okuma (non-repeatable read) |
| **Lost Update** | İki eş zamanlı read-modify-write döngüsünde birinin güncelleme kaybetmesi |
| **Write Skew** | İki transaction, bir koşulu kontrol edip birbirinin write'ını etkileyecek şekilde güncelleme yapması |
| **Phantom** | Bir transaction'ın write'ının diğerinin arama sorgusunu etkilemesi |
| **Snapshot Isolation** | Her transaction, başlangıç anındaki tutarlı snapshot'tan okur |
| **MVCC** | Multiversion Concurrency Control — satırların birden fazla versiyonunu saklama |
| **Serializability** | Transaction'ların sanki sıralı yürütülmüş gibi sonuç vermesi |
| **2PL** | Two-Phase Locking — shared/exclusive lock ile serializability sağlama |
| **SSI** | Serializable Snapshot Isolation — snapshot isolation üzerine çakışma tespiti |
| **2PC** | Two-Phase Commit — distributed transaction'larda atomicity sağlama protokolü |
| **Coordinator** | 2PC'de karar veren merkezi bileşen |
| **Participant** | 2PC'de transaction'a dahil olan database node'u |
| **XA Transaction** | Heterojen sistemlerde 2PC standardı |
| **In-Doubt** | Coordinator'dan karar bekleyen, ne yapacağını bilemeyen participant durumu |
| **Heuristic Decision** | Coordinator olmadan participant'ın tek taraflı karar vermesi (atomicity ihlal riski) |
| **Predicate Lock** | Bir arama koşulunu karşılayan tüm nesnelere ait lock |
| **Index-Range Lock** | Bir index aralığına ait lock (predicate lock'ın pratik yaklaşımı) |
| **Materializing Conflicts** | Phantom lock sorununu çözmek için yapay lock nesneleri oluşturma |
| **Exactly-Once Semantics** | Mesajın tam olarak bir kez işlenmesi garantisi |
| **Idempotent** | Aynı işlemin birden fazla kez uygulanmasının sonucu değiştirmemesi |
| **Read-Modify-Write Cycle** | Bir değeri okuyup değiştirip geri yazma döngüsü |
| **Compare-and-Set (CAS)** | Değer değişmemişse yazma işlemi yapan atomik operasyon |

## 13. Pratik Kod Örnekleri

In [ ]:
# Transaction kavramlarını Python ile simüle eden demo
# (Gerçek database bağlantısı olmadan kavramsal gösterim)

import threading
import time

# ============================================================
# Lost Update Problemi: Read-Modify-Write Cycle
# ============================================================

class UnsafeCounter:
    """Atomic olmayan increment — Lost Update'e açık"""
    def __init__(self, value=0):
        self.value = value
    
    def increment(self, thread_name):
        # 1. READ: Mevcut değeri oku
        current = self.value
        # Yapay gecikme - race condition'ı tetiklemek için
        time.sleep(0.001)
        # 2. MODIFY: Değeri hesapla
        new_value = current + 1
        # 3. WRITE: Geri yaz
        self.value = new_value
        print(f"{thread_name}: {current} -> {new_value} (yazdı)")

class SafeCounter:
    """Atomic increment — thread-safe (lock kullanarak)"""
    def __init__(self, value=0):
        self.value = value
        self._lock = threading.Lock()
    
    def increment(self, thread_name):
        with self._lock:  # Exclusive lock: sadece bir thread aynı anda
            current = self.value
            time.sleep(0.001)
            self.value = current + 1
            print(f"{thread_name}: {current} -> {self.value} (yazdı, güvenli)")


print("=" * 60)
print("UNSAFE COUNTER (Lost Update Riski)")
print("=" * 60)
unsafe = UnsafeCounter(0)
threads = []
for i in range(5):
    t = threading.Thread(target=unsafe.increment, args=(f"Thread-{i+1}",))
    threads.append(t)

for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"\nBeklenen sonuç: 5, Gerçek sonuç: {unsafe.value}")
if unsafe.value < 5:
    print("❌ LOST UPDATE oluştu!")

print("\n" + "=" * 60)
print("SAFE COUNTER (Atomic Increment)")
print("=" * 60)
safe = SafeCounter(0)
threads = []
for i in range(5):
    t = threading.Thread(target=safe.increment, args=(f"Thread-{i+1}",))
    threads.append(t)

for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"\nBeklenen sonuç: 5, Gerçek sonuç: {safe.value}")
if safe.value == 5:
    print("✅ Doğru sonuç! Lost update önlendi.")

In [ ]:
# ============================================================
# SQL ile Transaction ve Isolation Level Örnekleri
# ============================================================

# Aşağıdaki kod SQL syntax'ını göstermek amacıyla yazılmıştır.
# Gerçek bir database bağlantısı gerektirmez — sadece konsept gösterimi.

sql_examples = {
    "Basit Transaction": """
    BEGIN TRANSACTION;
    
    UPDATE accounts SET balance = balance - 100 WHERE user_id = 1;
    UPDATE accounts SET balance = balance + 100 WHERE user_id = 2;
    
    -- İki UPDATE başarılıysa:
    COMMIT;
    
    -- Hata oluşursa:
    -- ROLLBACK;
    """,
    
    "Isolation Level Ayarı": """
    -- PostgreSQL'de isolation level ayarlama
    BEGIN TRANSACTION ISOLATION LEVEL SERIALIZABLE;
    -- veya
    SET TRANSACTION ISOLATION LEVEL READ COMMITTED;
    """,
    
    "SELECT FOR UPDATE (Explicit Lock)": """
    -- Lost update önlemek için explicit lock
    BEGIN TRANSACTION;
    
    SELECT * FROM doctors
    WHERE on_call = TRUE AND shift_id = 1234
    FOR UPDATE;  -- Bu satırları kilitler, başka transaction'lar bekler
    
    UPDATE doctors
    SET on_call = FALSE
    WHERE name = 'Aaliyah' AND shift_id = 1234;
    
    COMMIT;
    """,
    
    "Atomic Update (Lost Update'i Önler)": """
    -- Ayrı bir read-modify-write döngüsü yerine atomik güncelleme
    -- Bu sorgu read-modify-write yerine tek atomic operasyondur
    UPDATE counters SET value = value + 1 WHERE key = 'page_views';
    """,
    
    "Compare-and-Set": """
    -- Eski değer hala aynıysa güncelle
    UPDATE wiki_pages
    SET content = 'Yeni içerik',
        version = version + 1
    WHERE id = 42
      AND content = 'Eski içerik';  -- Optimistic locking koşulu
    
    -- Etkilenen satır sayısı 0 ise başkası güncellemiş demektir
    -- Uygulama yeniden denemelidir.
    """,
    
    "Materializing Conflicts (Meeting Room)": """
    -- Phantom write skew'i önlemek için materializing conflicts
    BEGIN TRANSACTION;
    
    -- Önce lock tablosundaki oda-zaman dilimi satırını kilitle
    SELECT * FROM room_time_slots
    WHERE room_id = 123
      AND time_slot = '2025-01-01 12:00'
    FOR UPDATE;
    
    -- Örtüşen rezervasyon var mı kontrol et
    SELECT COUNT(*) FROM bookings
    WHERE room_id = 123
      AND end_time > '2025-01-01 12:00'
      AND start_time < '2025-01-01 13:00';
    
    -- Yoksa rezervasyon yap
    INSERT INTO bookings (room_id, start_time, end_time, user_id)
    VALUES (123, '2025-01-01 12:00', '2025-01-01 13:00', 666);
    
    COMMIT;
    """,
    
    "Exactly-Once Processing (Message Deduplication)": """
    -- Mesajı tam olarak bir kez işlemek için idempotent yaklaşım
    BEGIN TRANSACTION;
    
    -- 1. Mesaj daha önce işlendi mi?
    SELECT id FROM processed_messages WHERE message_id = 'msg-12345';
    
    -- 2. Eğer yoksa mesaj ID'sini kaydet
    INSERT INTO processed_messages (message_id, processed_at)
    VALUES ('msg-12345', NOW());
    
    -- 3. Asıl işlemi yap
    UPDATE inventory SET quantity = quantity - 1 WHERE product_id = 42;
    
    COMMIT;
    -- Commit başarılıysa mesajı broker'a acknowledge et
    """
}

for title, sql in sql_examples.items():
    print(f"\n{'='*60}")
    print(f"📝 {title}")
    print('='*60)
    print(sql)

In [ ]:
# ============================================================
# Write Skew Senaryosu Simülasyonu
# ============================================================

class Hospital:
    """Doktor nöbet takip sistemi — Write Skew senaryosu"""
    
    def __init__(self):
        self.doctors_on_call = {"Aaliyah": True, "Bryce": True}  # 2 doktor nöbette
        self._lock = threading.Lock()  # Explicit lock
    
    def go_off_call_UNSAFE(self, doctor_name):
        """UNSAFE: Write skew'e açık (snapshot isolation benzeri davranış)"""
        # Her iki transaction da bu kısmı eş zamanlı çalıştırır
        on_call_count = sum(v for v in self.doctors_on_call.values())
        
        # Gecikme ekle — race condition'ı tetikle
        time.sleep(0.01)
        
        if on_call_count >= 2:
            self.doctors_on_call[doctor_name] = False
            return f"{doctor_name} nöbetten ayrıldı (nöbetteki sayı: {on_call_count})"
        else:
            return f"{doctor_name} nöbetten ayrılamadı (yeterli doktor yok)"
    
    def go_off_call_SAFE(self, doctor_name):
        """SAFE: Explicit locking ile write skew önlenir"""
        with self._lock:  # SELECT FOR UPDATE gibi davranır
            on_call_count = sum(v for v in self.doctors_on_call.values())
            
            if on_call_count >= 2:
                self.doctors_on_call[doctor_name] = False
                return f"{doctor_name} nöbetten ayrıldı (güvenli)"
            else:
                return f"{doctor_name} nöbetten ayrılamadı (yeterli doktor yok)"


print("=" * 60)
print("UNSAFE: Write Skew Senaryosu")
print("=" * 60)
hospital_unsafe = Hospital()

results = []
def run_unsafe(doctor):
    results.append(hospital_unsafe.go_off_call_UNSAFE(doctor))

t1 = threading.Thread(target=run_unsafe, args=("Aaliyah",))
t2 = threading.Thread(target=run_unsafe, args=("Bryce",))
t1.start(); t2.start()
t1.join(); t2.join()

for r in results:
    print(r)

on_call = {k: v for k, v in hospital_unsafe.doctors_on_call.items() if v}
print(f"\nNöbetteki doktorlar: {list(on_call.keys())}")
if len(on_call) == 0:
    print("❌ WRITE SKEW! Kimse nöbette değil — kural ihlali!")


print("\n" + "=" * 60)
print("SAFE: Explicit Locking ile Write Skew Önlenir")
print("=" * 60)
hospital_safe = Hospital()

results_safe = []
def run_safe(doctor):
    results_safe.append(hospital_safe.go_off_call_SAFE(doctor))

t1 = threading.Thread(target=run_safe, args=("Aaliyah",))
t2 = threading.Thread(target=run_safe, args=("Bryce",))
t1.start(); t2.start()
t1.join(); t2.join()

for r in results_safe:
    print(r)

on_call_safe = {k: v for k, v in hospital_safe.doctors_on_call.items() if v}
print(f"\nNöbetteki doktorlar: {list(on_call_safe.keys())}")
if len(on_call_safe) >= 1:
    print("✅ Kural korundu! En az bir doktor hala nöbette.")

In [ ]:
# ============================================================
# MVCC (Multiversion Concurrency Control) Kavramı Simülasyonu
# ============================================================

class MVCCRow:
    """Bir database satırının birden fazla versiyonunu tutan MVCC yapısı"""
    
    def __init__(self, initial_value, txid=0):
        # Her versiyon: (txid, value, deleted_by)
        self.versions = [(txid, initial_value, None)]
    
    def write(self, new_value, writing_txid, deleting_old=True):
        """Yeni bir versiyon ekle. UPDATE = delete + insert"""
        if deleting_old:
            # Eski versiyonu 'deleted_by' ile işaretle
            old_txid, old_val, _ = self.versions[-1]
            self.versions[-1] = (old_txid, old_val, writing_txid)
        
        # Yeni versiyonu ekle
        self.versions.append((writing_txid, new_value, None))
        print(f"WRITE: txid={writing_txid}, yeni değer={new_value}")
    
    def read(self, reading_txid, in_progress_txids=None):
        """MVCC visibility rules uygulayarak görünür versiyonu döndür"""
        if in_progress_txids is None:
            in_progress_txids = set()
        
        # En yeni versiyondan geriye doğru görünür olanı bul
        visible_value = None
        for txid, value, deleted_by in reversed(self.versions):
            # Kural 1: Okuyan transaction başladığında bu txid zaten commit olmuş mu?
            if txid > reading_txid or txid in in_progress_txids:
                continue  # Görünmez: sonraki ya da hala devam eden transaction
            
            # Kural 2: Silindi mi? Silindiyse, silme transaction'ı daha önce commit edilmiş mi?
            if deleted_by is not None:
                if deleted_by <= reading_txid and deleted_by not in in_progress_txids:
                    continue  # Silindi ve silme görünür
            
            visible_value = value
            break
        
        return visible_value
    
    def show_versions(self):
        print("\nTüm Versiyonlar:")
        print(f"{'txid':>8} {'value':>10} {'deleted_by':>12}")
        print("-" * 35)
        for txid, value, deleted_by in self.versions:
            deleted_str = str(deleted_by) if deleted_by else "-"
            print(f"{txid:>8} {str(value):>10} {deleted_str:>12}")


print("=" * 60)
print("MVCC Örneği: Accounts tablosu, hesap 2 bakiyesi")
print("=" * 60)

# Başlangıç: txid=1 ile 500 dolar bakiye
account2 = MVCCRow(initial_value=500, txid=1)

print("\nBaşlangıç durumu: Hesap 2'nin bakiyesi = $500")

# Transaction 12 başlıyor (okuyacak)
reading_txid = 12
in_progress = {13}  # Transaction 13 henüz commit edilmedi

# Transaction 13, bakiyeyi 500'den 400'e düşürüyor ($100 transfer)
print("\nTransaction 13: $500 -> $400 yazıyor (henüz commit edilmedi)")
account2.write(new_value=400, writing_txid=13)

account2.show_versions()

# Transaction 12, kendi snapshot'ından okuyor
# Transaction 13 hala in-progress olduğu için görünmez
value_seen_by_tx12 = account2.read(reading_txid=12, in_progress_txids={13})
print(f"\nTransaction 12'nin gördüğü değer (txid=13 in-progress): ${value_seen_by_tx12}")
print("(Transaction 12, transaction 13'ün write'ını görmez → snapshot isolation!)")

# Şimdi transaction 13 commit ediyor
print("\nTransaction 13 commit etti.")

# Transaction 14, commit sonrası okuyabilir
value_seen_by_tx14 = account2.read(reading_txid=14, in_progress_txids=set())
print(f"Transaction 14'ün gördüğü değer: ${value_seen_by_tx14}")
print("(Transaction 14, transaction 13'ün write'ını görür)")

In [ ]:
# ============================================================
# Two-Phase Commit (2PC) Akış Simülasyonu
# ============================================================

import random

class DatabaseNode:
    """2PC'deki participant database node'u"""
    
    def __init__(self, node_id, failure_rate=0.0):
        self.node_id = node_id
        self.failure_rate = failure_rate
        self.prepared = False
        self.committed = False
    
    def prepare(self, txid):
        """Phase 1: Commit edebilir misiniz?"""
        if random.random() < self.failure_rate:
            print(f"  Node {self.node_id}: PREPARE [{txid}] → HAYIR (hata!)")
            return False
        
        self.prepared = True
        print(f"  Node {self.node_id}: PREPARE [{txid}] → EVET (commit yapabilirim)")
        return True
    
    def commit(self, txid):
        """Phase 2: Commit et"""
        if self.prepared:
            self.committed = True
            print(f"  Node {self.node_id}: COMMIT [{txid}] → Başarılı ✅")
    
    def abort(self, txid):
        """Phase 2: Geri al"""
        self.prepared = False
        print(f"  Node {self.node_id}: ABORT [{txid}] → Geri alındı 🔄")


class TransactionCoordinator:
    """2PC Coordinator"""
    
    def __init__(self):
        self.txid_counter = 1000
    
    def run_2pc(self, participants, description=""):
        txid = self.txid_counter
        self.txid_counter += 1
        
        print(f"\n{'='*60}")
        print(f"2PC Transaction ID: {txid} — {description}")
        print(f"{'='*60}")
        
        # ==========================================
        # PHASE 1: PREPARE
        # ==========================================
        print("\n📋 PHASE 1 — PREPARE")
        votes = {}
        for p in participants:
            votes[p.node_id] = p.prepare(txid)
        
        all_yes = all(votes.values())
        
        # ==========================================
        # Koordinatörün kararı (commit point)
        # ==========================================
        if all_yes:
            print(f"\n🏁 COMMIT POINT: Tüm node'lar EVET dedi → COMMIT kararı verildi")
            print(f"   (Bu karar transaction log'una yazıldı — artık geri dönüş yok!)")
        else:
            print(f"\n🛑 ABORT POINT: En az bir node HAYIR dedi → ABORT kararı verildi")
        
        # ==========================================
        # PHASE 2: COMMIT veya ABORT
        # ==========================================
        print(f"\n📋 PHASE 2 — {'COMMIT' if all_yes else 'ABORT'}")
        for p in participants:
            if all_yes:
                p.commit(txid)
            else:
                p.abort(txid)
        
        outcome = "COMMIT ✅" if all_yes else "ABORT ❌"
        print(f"\nSonuç: {outcome}")
        return all_yes


coordinator = TransactionCoordinator()

# Başarılı senaryo
participants = [
    DatabaseNode("DB-1", failure_rate=0.0),
    DatabaseNode("DB-2", failure_rate=0.0),
    DatabaseNode("DB-3", failure_rate=0.0),
]
coordinator.run_2pc(participants, "Tüm node'lar sağlıklı")

# Başarısız senaryo (bir node hata veriyor)
participants_with_failure = [
    DatabaseNode("DB-1", failure_rate=0.0),
    DatabaseNode("DB-2", failure_rate=1.0),  # Kesinlikle hata
    DatabaseNode("DB-3", failure_rate=0.0),
]
coordinator.run_2pc(participants_with_failure, "DB-2 hazır değil (abort senaryosu)")

## 14. Bölüm Özeti

Bu bölüm, transaction'ların database güvenilirliğindeki rolünü kapsamlı biçimde ele aldı:

### Temel Kavramlar

**Transaction'ların amacı**: Birden fazla read/write işlemini logical unit'e gruplandırarak uygulama programcısının partial failure konusundaki endişe yükünü azaltmak.

**ACID özellikleri**:
- **Atomicity**: Tüm işlemler başarılı olur ya da hiçbiri olmaz
- **Consistency**: Veri her zaman geçerli bir durumda kalır (uygulamanın sorumluluğu)
- **Isolation**: Eş zamanlı transaction'lar birbirini etkilemez
- **Durability**: Commit edilen veri kalıcıdır

### Isolation Seviyeleri ve Race Condition'lar

- **Read committed**: Dirty read ve dirty write'ı önler; read skew ve lost update'e karşı koruma sağlamaz
- **Snapshot isolation (MVCC)**: Read skew'i önler; lost update'i bazı durumlarda önler; write skew'e karşı koruma sağlamaz
- **Serializable isolation**: Tüm race condition'ları önler

### Serializability Yaklaşımları

- **Actual serial execution**: Basit ama tek thread ile sınırlı
- **Two-Phase Locking (2PL)**: Geleneksel pesimist yaklaşım; yüksek latency
- **Serializable Snapshot Isolation (SSI)**: Modern optimist yaklaşım; düşük latency, iyi concurrency

### Distributed Transactions

- **Atomic commitment problemi**: Tüm node'ların ya commit etmesi ya da abort etmesi gerekir
- **Two-Phase Commit (2PC)**: Coordinator + prepare/commit fazlarıyla atomicity sağlar
- **XA transactions**: Heterojen sistemlerde 2PC standardı; güvenilirlik sorunları mevcut
- **Database-internal distributed transactions**: NewSQL sistemleri XA sorunlarını aşar
- **Exactly-once semantics**: Distributed transaction yerine idempotent mesaj işleme ile sağlanabilir

---

*Bu notebook, "Designing Data-Intensive Applications" (Martin Kleppmann) kitabının Chapter 8: Transactions bölümünü kapsamaktadır.*